<a href="https://colab.research.google.com/github/marcelalozano27-ship-it/bsan6200-assignment5/blob/main/Colab%20Notebook/Assignment_5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Marcela Lozano
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid / Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [1]:
# ── Install packages (uncomment as needed) ──
!pip install -q chromadb sentence-transformers huggingface-hub python-dotenv

import os
import re
import pandas as pd
import requests
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
load_dotenv()


print("Imports working")

Imports working


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [2]:
# ── Load JD metadata ──
jd_metadata = pd.read_csv(
    "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/jd_metadata.csv"
)

print(jd_metadata)

                                             filename                Company  \
0                 jd_LAClippers_data_analyst_lead.txt            LA Clippers   
1         jd_alo_production_supplyplanning_intern.txt                    ALO   
2                jd_axs_associate_product_manager.txt                    AXS   
3          jd_ephemeris_financial_strategy_intern.txt             Ephemeris    
4                       jd_fedex_analytics_intern.txt                 Fedex    
5           jd_revolve_data_analyst_merchandising.txt                Revolve   
6       jd_roku_content_analytics_insights_intern.txt                   Roku   
7       jd_skechers_strategic_partnerships_intern.txt               Skechers   
8   jd_sony_insights_research_analytics_summerinte...                   Sony   
9      jd_tiktok_strategic_partner_manager_intern.txt                 TikTok   
10         jd_tinder_corporate_rotational_analyst.txt                 Tinder   
11  jd_universalmusicgroup_Brand_label_o

In [3]:
# ── Load JD documents and resume ──
from langchain_core.documents import Document
import requests

jd_documents = []

base_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/job_descriptions/"

for _, row in jd_metadata.iterrows():
    filename = row["filename"]
    url = base_url + filename

    text = requests.get(url).text

    doc = Document(
        page_content=text,
        metadata={
            "filename": filename,
            "company": row.get("company", ""),
            "title": row.get("title", ""),
            "source_url": row.get("source_url", ""),
            "date_collected": row.get("date_collected", ""),
            "doc_type": "job_description"
        }
    )

    jd_documents.append(doc)

print("Loaded job descriptions:", len(jd_documents))
print(jd_documents[0].metadata)
print(jd_documents[0].page_content[:500])

Loaded job descriptions: 13
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': '', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to optimization recommendations that increase conversion and yield. You will partner closely with Marketing, Sales and Finance stakeholders to translate data into clear actions and measurable outcomes.

T


In [4]:
resume_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/resume/resume.txt"

resume_text = requests.get(resume_url).text

resume_doc = Document(
    page_content=resume_text,
    metadata={
        "filename": "resume.txt",
        "doc_type": "resume"
    }
)

print("Loaded resume")
print(resume_doc.page_content[:500])

Loaded resume
MARCELA LOZANO
Los Angeles, CA
marcelalozano27@gmail.com
linkedin.com/in/marcelalozano

EDUCATION
Loyola Marymount University
M.S. Business Analytics, Expected Aug 2026
GPA: 4.0

University of Texas at Austin
Certification: Data Science and Data Management Systems
Skills: SQL, Python, Tableau, Google Cloud Platform

Loyola Marymount University
B.A. Economics, Minor Spanish
Magna Cum Laude, GPA: 3.89
Valedictorian of Economics Major

TECHNICAL SKILLS
Python
SQL
Tableau
Excel
Pandas
NumPy
scikit-l


In [5]:
# ── Preview sample content ──
all_documents = jd_documents + [resume_doc]

print("Total documents including Resume:", len(all_documents))
for i in range(3):
    print(f"\n--- Job Description {i} ---")
    print(jd_documents[i].metadata)
    print(jd_documents[i].page_content[:300])

Total documents including Resume: 14

--- Job Description 0 ---
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': '', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to o

--- Job Description 1 ---
{'filename': 'jd_alo_production_supplyplanning_intern.txt', 'company': '', 'title': 'Production & Supply Planning Intern', 'source_url': 'https://www.aloyoga.com/pages/careers', 'date_collected': '4/26/2026', 'doc_type': 'job_description'}

We are seeking a motivated and detail-oriented Production & Supply Planning Intern to 

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [7]:
# ── Strategy 1 ── Fixed size chunking

def chunk_text_fixed(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


fixed_chunks = []

for doc in all_documents:
    chunks = chunk_text_fixed(doc.page_content, chunk_size=800, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "fixed_size"
        })

print("Strategy 1 chunks:", len(fixed_chunks))
print(fixed_chunks[0]["text"][:300])

Strategy 1 chunks: 56
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [9]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=800, overlap_words=20):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())

            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return [c for c in chunks if len(c) > 50]


sentence_chunks = []

for doc in all_documents:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=800, overlap_words=20)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "sentence_aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(sentence_chunks[0]["text"][:300])

Strategy 2 chunks: 51
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [10]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    "strategy": ["fixed_size", "sentence_aware"],
    "chunk_size": [800, 800],
    "overlap": ["100 characters", "20 words"],
    "num_chunks": [len(fixed_chunks), len(sentence_chunks)]
})

comparison

,strategy,chunk_size,overlap,num_chunks
0,fixed_size,800,100 characters,56
1,sentence_aware,800,20 words,51


### Chunking Decision

**Which strategy did you choose?**  
Sentence Aware Chunking

**Why?**  
I'm using sentence aware chunking to preserve complete thoughts and sentences within the job descriptions. This helps improve the quality of retrieved information. Fixed size chunking splits important requirements across chunks and is therefore less meaningful for the analysis.  Sentence aware chunking (strategy 2) also provides slightly less chunks than strategy 1 making it a more efficient segmentation of the data.

**Final settings (chunk_size, overlap):**
chunk_size = 800
overlap = 20 words

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


client = chromadb.Client()
collection = client.create_collection(name="job_fit", get_or_create=True)

# use BEST chunks
all_chunks = sentence_chunks

# create embeddings + store
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=False)

# add to chroma
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=all_chunks,
    ids=[str(i) for i in range(len(texts))]
)

print("Vector DB created with", len(texts), "chunks")

In [ ]:
# ── Verify: run a test similarity search ──


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [ ]:
# ── Initialize LLM ──


In [ ]:
# ── Analysis 1: Skill Gap Report ──


In [ ]:
# ── Analysis 2: Keyword Alignment ──


In [ ]:
# ── Analysis 3: Fit Summary ──


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot version of your chosen analysis ──


In [ ]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [ ]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*